https://aws.amazon.com/blogs/machine-learning/build-a-contextual-text-and-image-search-engine-for-product-recommendations-using-amazon-bedrock-and-amazon-opensearch-serverless/

In [9]:
pip install pandas pillow opencv-python ultralytics scikit-learn torch

Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
import os
import shutil
from PIL import Image
import random
from sklearn.model_selection import train_test_split
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import torch

class FashionDatasetProcessor:
    def __init__(self, base_path, styles_csv_path, output_path):
        self.base_path = base_path
        # Modified CSV reading with error handling
        try:
            self.styles_df = pd.read_csv(styles_csv_path, encoding='utf-8')
        except:
            try:
                self.styles_df = pd.read_csv(styles_csv_path, encoding='latin1')
            except:
                # Using on_bad_lines instead of error_bad_lines
                self.styles_df = pd.read_csv(styles_csv_path, encoding='utf-8', on_bad_lines='skip')
                print("Some rows were skipped due to parsing errors")
        
        self.output_path = output_path
        self.class_mapping = {}
        
        # Print dataset info
        print(f"Total rows loaded: {len(self.styles_df)}")
        print("\nColumns in dataset:")
        print(self.styles_df.columns.tolist())
        
        # Clean the data
        self.clean_dataset()
    
    def clean_dataset(self):
        """Clean the dataset by removing rows with missing values"""
        initial_len = len(self.styles_df)
        
        # Remove rows with missing articleType
        self.styles_df = self.styles_df.dropna(subset=['articleType'])
        
        # Remove rows where id is not numeric
        self.styles_df['id'] = pd.to_numeric(self.styles_df['id'], errors='coerce')
        self.styles_df = self.styles_df.dropna(subset=['id'])
        self.styles_df['id'] = self.styles_df['id'].astype(int)
        
        final_len = len(self.styles_df)
        print(f"\nRows removed during cleaning: {initial_len - final_len}")
        print(f"Final number of rows: {final_len}")
        
        # Print unique article types
        print("\nUnique article types:")
        print(self.styles_df['articleType'].unique())

    def create_directory_structure(self):
        """Create the necessary directories for YOLO training"""
        for split in ['train', 'val', 'test']:
            os.makedirs(os.path.join(self.output_path, 'images', split), exist_ok=True)
            os.makedirs(os.path.join(self.output_path, 'labels', split), exist_ok=True)
    
    def create_class_mapping(self):
        """Create a mapping of unique article types to class indices"""
        unique_articles = sorted(self.styles_df['articleType'].unique())
        self.class_mapping = {article: idx for idx, article in enumerate(unique_articles)}
        
        # Save class mapping
        with open(os.path.join(self.output_path, 'classes.txt'), 'w') as f:
            for article, idx in self.class_mapping.items():
                f.write(f'{article},{idx}\n')
                
    def create_yolo_annotation(self, img_path, article_type):
        """Create YOLO format annotation"""
        try:
            img = Image.open(img_path)
            width, height = img.size
            
            # Create a basic bounding box (80% of image size)
            box_w = 0.8
            box_h = 0.8
            x_center = 0.5
            y_center = 0.5
            
            class_id = self.class_mapping[article_type]
            return f"{class_id} {x_center} {y_center} {box_w} {box_h}"
        except Exception as e:
            print(f"Error processing image {img_path}: {e}")
            return None

    def process_dataset(self):
        """Process the dataset and create train/val/test splits"""
        # Create train/val/test splits
        image_ids = self.styles_df['id'].tolist()
        train_ids, test_val_ids = train_test_split(image_ids, test_size=0.3, random_state=42)
        val_ids, test_ids = train_test_split(test_val_ids, test_size=0.5, random_state=42)
        
        splits = {
            'train': train_ids,
            'val': val_ids,
            'test': test_ids
        }
        
        processed_count = 0
        total_images = len(image_ids)
        
        for split, ids in splits.items():
            for img_id in ids:
                row = self.styles_df[self.styles_df['id'] == img_id].iloc[0]
                
                # Source image path
                src_img_path = os.path.join(self.base_path, 'images', f'{img_id}.jpg')
                
                if not os.path.exists(src_img_path):
                    continue
                
                # Destination paths
                dst_img_path = os.path.join(self.output_path, 'images', split, f'{img_id}.jpg')
                dst_label_path = os.path.join(self.output_path, 'labels', split, f'{img_id}.txt')
                
                # Copy image
                shutil.copy2(src_img_path, dst_img_path)
                
                # Create and save annotation
                annotation = self.create_yolo_annotation(src_img_path, row['articleType'])
                if annotation:
                    with open(dst_label_path, 'w') as f:
                        f.write(annotation)
                
                processed_count += 1
                if processed_count % 1000 == 0:
                    print(f"Processed {processed_count}/{total_images} images")
    
    def create_yaml_config(self):
        """Create YAML configuration file for YOLOv8"""
        yaml_content = f"""
path: {self.output_path}
train: images/train
val: images/val
test: images/test

names:
{chr(10).join(f'  {i}: {name}' for name, i in self.class_mapping.items())}
"""
        with open(os.path.join(self.output_path, 'data.yaml'), 'w') as f:
            f.write(yaml_content)

def prepare_fashion_dataset(base_path, output_path='fashion_yolo_dataset'):
    """Prepare the fashion dataset with error handling and verbose output"""
    base_path = Path(base_path)
    styles_csv_path = base_path / 'styles.csv'
    
    # Check if paths exist
    if not base_path.exists():
        raise FileNotFoundError(f"Dataset directory not found at {base_path}")
    
    if not styles_csv_path.exists():
        raise FileNotFoundError(f"styles.csv not found at {styles_csv_path}")
    
    # Check if images directory exists
    images_dir = base_path / 'images'
    if not images_dir.exists():
        raise FileNotFoundError(f"Images directory not found at {images_dir}")
    
    print("Initializing dataset processor...")
    processor = FashionDatasetProcessor(
        base_path=str(base_path),
        styles_csv_path=str(styles_csv_path),
        output_path=output_path
    )
    
    print("\nCreating directory structure...")
    processor.create_directory_structure()
    
    print("Creating class mapping...")
    processor.create_class_mapping()
    
    print("Processing dataset...")
    processor.process_dataset()
    
    print("Creating YAML config...")
    processor.create_yaml_config()
    
    return processor

def train_fashion_model(yaml_path, epochs=100, img_size=640, batch_size=16):
    """Train YOLOv8 model on the processed dataset"""
    # Load a pretrained YOLOv8 model
    model = YOLO('yolov8n.pt')
    
    # Train the model
    results = model.train(
        data=yaml_path,
        epochs=epochs,
        imgsz=img_size,
        batch=batch_size,
        name='fashion_model',
        patience=20,  # Early stopping patience
        save=True,  # Save best model
        device='0' if torch.cuda.is_available() else 'cpu'  # Use GPU if available
    )
    
    return results

def test_model(model_path, image_path, conf_threshold=0.25):
    """Test the trained model on an image"""
    model = YOLO(model_path)
    
    results = model.predict(
        source=image_path,
        conf=conf_threshold,
        save=True
    )
    
    return results

In [ ]:
# Example usage
if __name__ == "__main__":
    try:
        # 1. Prepare dataset
        base_path = "SODA files/kaggle"  # Replace with your dataset path
        processor = prepare_fashion_dataset(base_path)
        
        # 2. Train model
        yaml_path = os.path.join('fashion_yolo_dataset', 'data.yaml')
        results = train_fashion_model(yaml_path)
        
        # 3. Test model (optional)
        test_image_path = "SODA files/mariners.jpg"  # Replace with your test image
        model_path = "runs/train/fashion_model/weights/best.pt"
        test_results = test_model(model_path, test_image_path)
        
        print("Training and testing completed successfully!")
        
    except Exception as e:
        print(f"Error: {e}")

Initializing dataset processor...
Some rows were skipped due to parsing errors
Total rows loaded: 44424

Columns in dataset:
['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName']

Rows removed during cleaning: 0
Final number of rows: 44424

Unique article types:
['Shirts' 'Jeans' 'Watches' 'Track Pants' 'Tshirts' 'Socks' 'Casual Shoes' 'Belts' 'Flip Flops' 'Handbags' 'Tops' 'Bra' 'Sandals' 'Shoe Accessories' 'Sweatshirts' 'Deodorant' 'Formal Shoes' 'Bracelet' 'Lipstick' 'Flats' 'Kurtas' 'Waistcoat' 'Sports Shoes' 'Shorts' 'Briefs' 'Sarees' 'Perfume and Body Mist' 'Heels'
 'Sunglasses' 'Innerwear Vests' 'Pendant' 'Nail Polish' 'Laptop Bag' 'Scarves' 'Rain Jacket' 'Dresses' 'Night suits' 'Skirts' 'Wallets' 'Blazers' 'Ring' 'Kurta Sets' 'Clutches' 'Shrug' 'Backpacks' 'Caps' 'Trousers' 'Earrings' 'Camisoles' 'Boxers' 'Jewellery Set' 'Dupatta' 'Capris' 'Lip Gloss' 'Bath Robe' 'Mufflers'
 'Tunics' 'Jackets' 'Trunk' 'Lo

100%|█████████████████████████████████████████████████████████████████████████████| 6.25M/6.25M [00:00<00:00, 55.5MB/s]


New https://pypi.org/project/ultralytics/8.3.146 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.145  Python-3.11.5 torch-2.2.0+cu121 CPU (11th Gen Intel Core(TM) i7-1185G7 3.00GHz)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=fashion_yolo_dataset\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=fashion_model, nbs=64, nms=False

100%|███████████████████████████████████████████████████████████████████████████████| 755k/755k [00:00<00:00, 37.7MB/s]

Overriding model.yaml nc=80 with nc=143

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralyti

...

train: Scanning C:\Users\methvenr\fashion_yolo_dataset\labels\train... 31092 images, 0 backgrounds, 0 corrupt: 100%|███


train: New cache created: C:\Users\methvenr\fashion_yolo_dataset\labels\train.cache
val: Fast image access  (ping: 0.20.1 ms, read: 40.227.7 MB/s, size: 14.7 KB)


val: Scanning C:\Users\methvenr\fashion_yolo_dataset\labels\val... 6663 images, 0 backgrounds, 0 corrupt: 100%|████████


val: New cache created: C:\Users\methvenr\fashion_yolo_dataset\labels\val.cache
Plotting labels to runs\detect\fashion_model\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: SGD(lr=0.01, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to runs\detect\fashion_model
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      1/100         0G      1.838      5.292      2.341         47        640:  11%|█         | 210/1944 [17:41:23<10:0